In [ ]:
#@title DeFooocus
#@markdown **Launch the interface DeFocus (Fooocus fork)** | You need to connect with T4/A100/V100
#@markdown ****
#@markdown *Attention!* When working in the interface with the FaceSwap and CPDS controlnet, crashes are possible; it is also recommended to work in *Extreme speed* mode for additional stability. When working with the ImagePrompt and PyraCanny controls, 85% of the work will be stable.
#@markdown ****

print("[DeFooocus] Preparing ...")

theme = "dark" #@param ["dark", "light"]
preset = "deafult" #@param ["deafult", "realistic", "anime", "lcm", "sai", "turbo", "lighting", "hypersd", "playground_v2.5", "dpo", "spo", "sd1.5"]
advenced_args = "--share --attention-split --always-high-vram --disable-offload-from-vram --all-in-fp16" #@param {type: "string"}

if preset != "deafult":
  args = f"{advenced_args} --theme {theme} --preset {preset}"
else:
  args = f"{advenced_args} --theme {theme}"

!pip install -q pygit2            # latest version has Python 3.13 wheels
%cd /content
![ -d DeFooocus ] || git clone https://github.com/ehristoforu/DeFooocus.git
%cd /content/DeFooocus

# Patch DeFooocus's bundled BLIP for transformers 4.5x
import re
_p = "/content/DeFooocus/extras/BLIP/models/med.py"
_src = open(_p).read()

_src = re.sub(
    r"from transformers\.modeling_utils import \((.*?)\)",
    '''from transformers.modeling_utils import PreTrainedModel
from transformers import GenerationMixin
from transformers.pytorch_utils import apply_chunking_to_forward, prune_linear_layer
try:
    from transformers.pytorch_utils import find_pruneable_heads_and_indices
except ImportError:
    def find_pruneable_heads_and_indices(heads, n_heads, head_size, already_pruned_heads):
        mask = torch.ones(n_heads, head_size)
        heads = set(heads) - already_pruned_heads
        for head in heads:
            head = head - sum(1 if h < head else 0 for h in already_pruned_heads)
            mask[head] = 0
        mask = mask.view(-1).contiguous().eq(1)
        index = torch.arange(len(mask))[mask].long()
        return heads, index''',
    _src, count=1, flags=re.DOTALL)

# transformers >= 4.50 no longer gives every model .generate(); BLIP's decoder needs it
_src = _src.replace("class BertLMHeadModel(BertPreTrainedModel):",
                    "class BertLMHeadModel(BertPreTrainedModel, GenerationMixin):")
open(_p, "w").write(_src)

# drop pins that can't install on Python 3.13
!sed -i '/^pygit2/d;/^numpy/d' requirements_versions.txt
!pip install -r requirements_versions.txt     # no -q, so errors are visible

# keep transformers on 4.x so the CLIP weights load correctly
!pip install -q "transformers>=4.46,<5"

# patch gradio 3.x for NumPy 2
!sed -i 's/np\.bool8/np.bool_/g' "$(python -c 'import gradio,os;print(os.path.dirname(gradio.__file__))')/processing_utils.py"

# NumPy 1.x compatibility for gradio 3.x (NumPy 2 removed these names)
import numpy, os
_site = os.path.dirname(os.path.dirname(numpy.__file__))

open(os.path.join(_site, "numpy2_compat.py"), "w").write('''
import numpy as np

def _obj2sctype(rep, default=None):
    if isinstance(rep, type) and issubclass(rep, np.generic):
        return rep
    try:
        return np.dtype(rep).type
    except Exception:
        return default

_aliases = {
    "float_": np.float64, "bool8": np.bool_, "int0": np.intp, "uint0": np.uintp,
    "complex_": np.complex128, "cfloat": np.complex128, "unicode_": np.str_,
    "string_": np.bytes_, "str0": np.str_, "bytes0": np.bytes_, "object0": np.object_,
    "void0": np.void, "longfloat": np.longdouble, "singlecomplex": np.complex64,
    "Inf": np.inf, "NaN": np.nan, "infty": np.inf, "PINF": np.inf, "NINF": -np.inf,
    "obj2sctype": _obj2sctype,
    "issctype": lambda rep: isinstance(rep, type) and issubclass(rep, np.generic),
    "sctype2char": lambda s: np.dtype(s).char,
    "find_common_type": lambda a, s=(): np.result_type(*a, *s),
    "round_": np.round, "product": np.prod, "cumproduct": np.cumprod,
    "sometrue": np.any, "alltrue": np.all,
    "asfarray": lambda a, dtype=np.float64: np.asarray(a, dtype=dtype),
}
for _k, _v in _aliases.items():
    if not hasattr(np, _k):
        setattr(np, _k, _v)
''')

open(os.path.join(_site, "zz_numpy2_compat.pth"), "w").write("import numpy2_compat\n")
!python -c "import numpy as np; print(np.obj2sctype(np.uint8), np.float_, np.bool8)"

print("[DeFooocus] Starting ...")
!python entry_with_update.py $args

[DeFooocus] Preparing ...
/content
/content/DeFooocus
<class 'numpy.uint8'> <class 'numpy.float64'> <class 'numpy.bool'>
[DeFooocus] Starting ...
Already up-to-date
Update succeeded.
[System ARGV] ['entry_with_update.py', '--share', '--attention-split', '--always-high-vram', '--disable-offload-from-vram', '--all-in-fp16', '--theme', 'dark']
Python 3.13.15 (main, Aug  6 2026, 11:06:22) [GCC 13.3.0]
Fooocus version: 0.2
Total VRAM 14913 MB, total RAM 12976 MB
Forcing FP16.
Set vram state to: HIGH_VRAM
Device: cuda:0 Tesla T4 : native
VAE dtype: torch.float32
Using split optimization for cross attention
2026-10-05 09:37:36.524680: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX512F FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
Refiner unloaded.
model_type EPS
UNet ADM Dimension 2816
IMPORTANT: You 